In [19]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor
import threading
from pprint import pprint

In [22]:
API_KEY = '9e6ba51d1c664563a829072ba40ef69c'
LAYER_ID = 98970
BASE_URL = 'https://koordinates.com/services/query/v1/vector.json'
CODE_COL = 'SA22019_V1_00'

MAX_WORKERS = 5
_thread_local = threading.local()

In [3]:
def _get_session():
    if not hasattr(_thread_local, "session"):
        _thread_local.session = requests.Session()
    return _thread_local.session

In [23]:
def find_area_code(lon_lat):
    lon, lat = lon_lat
    params = {
        "key": API_KEY,
        "layer": LAYER_ID,
        "x": lon,
        "y": lat,
        "max_results": 1,
        "radius": 1,  # small radius — for a polygon layer, a point inside returns distance 0
    }
    resp = _get_session().get(BASE_URL, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    features = data["vectorQuery"]["layers"][str(LAYER_ID)]["features"]
    if not features:
        return None
    area_code = features[0]["properties"][CODE_COL]
    return area_code

In [32]:
listings_chch = pd.read_csv('../data_d3/listings_chch.csv')

In [25]:
pairs = list(zip(listings_chch.longitude, listings_chch.latitude))
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    results = list(executor.map(find_area_code, pairs))

In [33]:
listings_chch['sa2_code'] = results
listings_chch['sa2_code'] = listings_chch['sa2_code'].astype(int)

In [38]:
listings_chch.to_csv('../data_d3/listings_chch_codes.csv')